In [1]:
from utils import *
from CS_feature_extractor import *
from CS_based_early_stopping import *

[nltk_data] Downloading package punkt to /home/wxr9et/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/wxr9et/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# How to run the code to get Acc and # of API calls

In [2]:
DATA_DIR = "../data/Evaluation_CoTs/sample_data/"
input_file_path = os.path.join(DATA_DIR, 'GSM8K_GPT4o_mini.csv')
df_llama = pd.read_csv(input_file_path).reset_index(drop=True)

In [3]:
df_test = df_llama.sample(100).reset_index(drop=True) # get partial data for test

In [4]:
df_test  # Please put your question and CoT in the given format

,Name,Category,Question,Correct Answer,CoT_0,Final Answer_0,Instruction Violation_0,CoT_1,Final Answer_1,Instruction Violation_1,...,CoT_37,Final Answer_37,Instruction Violation_37,CoT_38,Final Answer_38,Instruction Violation_38,CoT_39,Final Answer_39,Instruction Violation_39,Prompt_File
0,GSM8K_test,Math,A wooden bridge can carry no more than 5000 po...,83,Step 1: Identify the given information - The m...,83,"[(0, 0)]",Step 1: Identify the weight limit of the bridg...,83,"[(0, 0)]",...,Step 1: Identify the given information - The m...,83,"[(0, 0)]",Step 1: Identify the given information - The m...,83,"[(0, 0)]",Step 1: Identify the given information - The m...,83,"[(0, 0)]",few_CoT.json
1,GSM8K_test,Math,Lee used to be able to run the 400-meter hurdl...,36,Step 1: Identify the given information - Lee's...,36.36,"[(0, 0)]",Step 1: Identify the given information - Lee's...,36,"[(0, 0)]",...,Step 1: Identify the given information - Lee r...,36,"[(0, 0)]",Step 1: Identify the given information - Lee's...,36,"[(0, 0)]",Step 1: Identify the given information - Lee's...,36,"[(0, 0)]",few_CoT.json
2,GSM8K_test,Math,Melanie is a door-to-door saleswoman. She sold...,18,Step 1: Define the variables - Let x be the to...,18,"[(0, 0)]",Step 1: Define the variables and understand th...,18,"[(0, 0)]",...,Step 1: Define the variables - Let x be the to...,18,"[(0, 0)]",Step 1: Let x be the total number of vacuum cl...,18,"[(0, 0)]",Step 1: Define the variables - Let x be the to...,33,"[(0, 0)]",few_CoT.json
3,GSM8K_test,Math,Kylar went to the store to buy glasses for his...,64,Step 1: Identify the cost structure for the gl...,64,"[(0, 0)]",Step 1: Identify the cost of the glasses - The...,64,"[(0, 0)]",...,Step 1: Identify the cost of the first glass a...,64,"[(0, 0)]",Step 1: Identify the cost of one glass - The f...,64,"[(0, 0)]",Step 1: Identify the cost structure for the gl...,64,"[(0, 0)]",few_CoT.json
4,GSM8K_test,Math,Dana can run at a rate of speed four times fas...,18,Step 1: Identify the given information - Dana ...,18 miles,"[(0, 0)]",Step 1: Identify the given information - Dana ...,18 miles,"[(0, 0)]",...,Step 1: Identify the given information - Dana ...,18 miles,"[(0, 0)]",Step 1: Identify the given information - Dana ...,18 miles,"[(0, 0)]",Step 1: Identify the given information - Dana ...,18 miles,"[(0, 0)]",few_CoT.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,GSM8K_test,Math,Christina is planning a birthday party and nee...,24,Step 1: Identify the given information - Chris...,18,"[(0, 0)]",Step 1: Identify the given information - Chris...,18,"[(0, 0)]",...,Step 1: Identify the given information - Chris...,18,"[(0, 0)]",Step 1: Identify the given information - Chris...,18,"[(0, 0)]",Step 1: Identify the given information - Chris...,18,"[(0, 0)]",few_CoT.json
96,GSM8K_test,Math,If Marcy works for the same company for 40 yea...,25000,Step 1: Identify the given information - Marcy...,75000,"[(0, 0)]",Step 1: Identify the given information - Marcy...,75000,"[(0, 0)]",...,Step 1: Identify the given information - Marcy...,75000,"[(0, 0)]",Step 1: Identify the given information - Marcy...,75000,"[(0, 0)]",Step 1: Identify the given information - Marcy...,"$75,000","[(0, 0)]",few_CoT.json
97,GSM8K_test,Math,Aleena subscribed to a streaming service that ...,1596,Step 1: Identify the monthly charge for the st...,1596,"[(0, 0)]",Step 1: Identify the monthly charge for the st...,1596,"[(0, 0)]",...,Step 1: Identify the given information - Month...,1596,"[(0, 0)]",Step 1: Identify the monthly charge for the st...,1596,"[(0, 0)]",Step 1: Identify the given information - Month...,$1596,"[(0, 0)]",few_CoT.json
98,GSM8K_test,Math,There are twice as many boys as girls at Dr. W...,36,Step 1: Identify the given information - There...,36,"[(0, 0)]",Step 1: Identify the given information - There...,36,"[(0, 0)]",...,Step 1: Identify the given information - Numbe...,36,"[(0, 0)]",Step 1: Identify the

In [5]:
df_test['Model'] = 'GPT4o'

In [6]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] 
# This includes total 10 features introduced in the paper; please see extract features for more details
data = extract_feature(df_test,feature_li)

jaccard with bigram time cost: 5.725274085998535s
jaccard with aggregation time cost: 56.0719940662384s


100%|██████████| 100/100 [00:05<00:00, 19.85it/s]


In [7]:
pd.DataFrame(data).head(5) # data is saved in json format

,id,Name,Model,correct answer,CoT answers,Correctness,SIM_AC_AGG,SIM_INPUT,DIF_IV,SIM_AC_BIGRAM,STEP_COUNT,SIM_COT_AGG,QUA_IM,LEN,STEP_COHERENCE,SIM_COT_BIGRAM
0,0,GSM8K_test,GPT4o,83,"[83.0, 83.0, 83.0, 83.0, 83.0, 83.0, 83.0, 83....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.2857142857142857, 0.36250000000000004, 0.29...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[6, 7, 5, 6, 5, 5, 6, 5, 6, 6, 6, 7, 7, 5, 6, ...","[0.0, 0.5581395348837209, 0.6288659793814433, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.21819460233253335, 0.30771821372227054, 0.2...","[0.0, 0.5581395348837209, 0.6666666666666667, ..."
1,1,GSM8K_test,GPT4o,36,"[36.4, 36.0, 36.0, 36.0, 36.4, 36.0, 36.0, 36....","[0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, ...","[0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, ...","[0.23863636363636365, 0.30952380952380953, 0.2...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, ...","[5, 6, 4, 4, 7, 4, 4, 4, 7, 4, 4, 4, 4, 6, 4, ...","[0.0, 0.5806451612903225, 0.5454545454545454, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...","[0.35833333333333334, 0.23312864831354213, 0.2...","[0.0, 0.5806451612903225, 0.6144578313253012, ..."
2,2,GSM8K_test,GPT4o,18,"[18.0, 18.0, 18.0, 18.0, 36.0, 18.0, 24.0, 18....","[1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, ...","[0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, ...","[0.29166666666666663, 0.2710280373831776, 0.28...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, ...","[10, 13, 13, 12, 10, 8, 9, 9, 10, 10, 10, 10, ...","[0.0, 0.631578947368421, 0.7, 0.62686567164179...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.4291433682701758, 0.40195283975701757, 0.43...","[0.0, 0.631578947368421, 0.75, 0.6637168141592..."
3,3,GSM8K_test,GPT4o,64,"[64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.25, 0.2816901408450704, 0.26315789473684215...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[7, 6, 7, 7, 7, 6, 6, 7, 5, 7, 5, 7, 6, 5, 8, ...","[0.0, 0.5897435897435898, 0.6235294117647059, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.23017485891805312, 0.2458162752280399, 0.33...","[0.0, 0.5897435897435898, 0.6911764705882353, ..."
4,4,GSM8K_test,GPT4o,18,"[18 miles, 18 miles, 18.0, 18 miles, 18 miles,...","[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","[0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, ...","[0.3292682926829268, 0.33333333333333337, 0.35...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, ...","[5, 6, 7, 6, 6, 7, 6, 8, 6, 6, 8, 9, 7, 9, 4, ...","[0.0, 0.7294117647058824, 0.6597938144329897, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ...","[0.38528880510012586, 0.31891502848889186, 0.2...","[0.0, 0.7294117647058824, 0.6702127659574468, ..."


In [8]:
df_processed = pd.DataFrame(data)

In [9]:
df_processed = calculate_SC_correctness(df_processed)

# Calculate Early Stopping Correctness with a specific window size
window_size = 5  # Define your window size
df_processed = calculate_ES_correctness(df_processed, window_size)

# Calculate Adaptive Consensus Correctness
df_processed = calculate_ASC_correctness(df_processed)

ES execution time: 0.0032 seconds
ASC execution time: 0.0254 seconds


In [10]:
df_processed.head() 

,id,Name,Model,correct answer,CoT answers,Correctness,SIM_AC_AGG,SIM_INPUT,DIF_IV,SIM_AC_BIGRAM,...,SIM_COT_AGG,QUA_IM,LEN,STEP_COHERENCE,SIM_COT_BIGRAM,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps
0,0,GSM8K_test,GPT4o,83,"[83.0, 83.0, 83.0, 83.0, 83.0, 83.0, 83.0, 83....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.2857142857142857, 0.36250000000000004, 0.29...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",...,"[0.0, 0.5581395348837209, 0.6288659793814433, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.21819460233253335, 0.30771821372227054, 0.2...","[0.0, 0.5581395348837209, 0.6666666666666667, ...",1,1,5,1,4
1,1,GSM8K_test,GPT4o,36,"[36.4, 36.0, 36.0, 36.0, 36.4, 36.0, 36.0, 36....","[0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, ...","[0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, ...","[0.23863636363636365, 0.30952380952380953, 0.2...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, ...",...,"[0.0, 0.5806451612903225, 0.5454545454545454, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, ...","[0.35833333333333334, 0.23312864831354213, 0.2...","[0.0, 0.5806451612903225, 0.6144578313253012, ...",1,1,24,1,25
2,2,GSM8K_test,GPT4o,18,"[18.0, 18.0, 18.0, 18.0, 36.0, 18.0, 24.0, 18....","[1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, ...","[0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, ...","[0.29166666666666663, 0.2710280373831776, 0.28...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, ...",...,"[0.0, 0.631578947368421, 0.7, 0.62686567164179...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.4291433682701758, 0.40195283975701757, 0.43...","[0.0, 0.631578947368421, 0.75, 0.6637168141592...",1,1,19,1,4
3,3,GSM8K_test,GPT4o,64,"[64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64.0, 64....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.25, 0.2816901408450704, 0.26315789473684215...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",...,"[0.0, 0.5897435897435898, 0.6235294117647059, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.23017485891805312, 0.2458162752280399, 0.33...","[0.0, 0.5897435897435898, 0.6911764705882353, ...",1,1,5,1,4
4,4,GSM8K_test,GPT4o,18,"[18 miles, 18 miles, 18.0, 18 miles, 18 miles,...","[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, ...","[0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, ...","[0.3292682926829268, 0.33333333333333337, 0.35...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, ...",...,"[0.0, 0.7294117647058824, 0.6597938144329897, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ...","[0.38528880510012586, 0.31891502848889186, 0.2...","[0.0, 0.7294117647058824, 0.6702127659574468, ...",0,0,8,0,7


In [14]:
# TO DO 1: Demonstrate 1: fit model with one dataset to get the coefficient; use coefficient with another dataset
# 2: Test code to run CoT (ADD prompt file to run CoT)
# 3: Write doc on how to run the python file. (special explanations for feature extraction)

In [16]:
feature_li = ['LEN', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE']  # We can take less if for test
df_confidence_scores,coefs = trained_LR_model(df_processed, feature_li, report_auroc=False,train_mode = True) # Note that we first try dataset to get the coefficient

Optimization terminated successfully.
         Current function value: 0.505883
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            Correctness   No. Observations:                 2800
Model:                          Logit   Df Residuals:                     2791
Method:                           MLE   Df Model:                            8
Date:                Tue, 04 Feb 2025   Pseudo R-squ.:                  0.1664
Time:                        14:37:08   Log-Likelihood:                -1416.5
converged:                       True   LL-Null:                       -1699.3
Covariance Type:            nonrobust   LLR p-value:                5.850e-117
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const             -0.9110      0.437     -2.084      0.037      -1.768      -0.054
LEN              

In [17]:
coefs

array([-0.91104471,  0.50800413, -3.09401477,  3.891465  ,  0.59170337,
        1.81612778, -5.17407762, -0.15324067,  5.8218749 ])

In [25]:
df_llama_confidence_scores.head()

,id,Name,Model,correct answer,CoT answers,Correctness,MATH_TERM_DENSITY,STEP_COHERENCE,SIM_AC_AGG,SIM_COT_AGG,...,DIF_IV,QUA_IM,STEP_COUNT,SIM_AC_BIGRAM,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps,confidence_score
0,40,BigBench_easy,llama3,C,"[D, C, A, E, E, C, E, A, D, A, C, E, E, D, D, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.5, 0.37209302325581395, 0.45033112582781454...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...",0,0,40,0,40,"[0.23638707935418551, 0.10582059028380454, 0.1..."
1,7,MathQA_challenge_test,llama3,c,"[E, C, E, E, E, E, E, E, C, C, D, D, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.012269938650306749, 0.02030456852791878, 0....","[0, 0, 0, 0, 0.32393939393939397, 0, 0, 0.2148...","[0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, ...","[0.5, 0.3652173913043478, 0.38, 0.254545454545...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 1, 3, 0, 0, 5, 0, 0, 0, 0, 0, 0, 3, ...","[0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, ...",0,0,7,0,7,"[0.18175323341013105, 0.06319644962629163, 0.1..."
2,10,MathQA_dev,llama3,b,"[E, A, A, E, E, E, E, E, E, E, A, E, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.005050505050505051, 0.011764705882352941, 0...","[0.2361111111111111, 0.22580645161290322, 0, 0...","[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, ...","[0.5, 0.6931818181818181, 0.6274509803921569, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 0, 0, 0, 2, 0, 0, 0, 2, 2, 2, 0, 2, 0, ...","[0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, ...",0,0,8,0,10,"[0.18299950629311443, 0.41321052181956053, 0.4..."
3,44,MathQA_challenge_test,llama3,a,"[E, E, E, D, E, E, E, E, E, E, E, E, E, E, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.011764705882352941, 0.011494252873563218, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0.32205919503079744, ...","[0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.5, 0.38497652582159625, 0.296137339055794, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,9,0,7,"[0.17491185964273384, 0.18835863334431088, 0.1..."
4,33,BigBench_easy,llama3,D,"[B, E, C, A, C, C, C, E, A, E, C, A, E, E, C, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.5, 0.45112781954887216, 0.3549382716049383,...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, ...",0,0,40,0,40,"[0.24724127099954366, 0.18596011704801862, 0.0..."


In [26]:
N = 5
threshold = 0.5

# Applying early stopping mechanism
df_final = CS_early_stopping(df=df_llama_confidence_scores, threshold=threshold, N=N)

SC_ACC : 0.2
ES_ACC : 0.2
CS_ACC : 0.3333333333333333
SC_Avg_Steps : 40
ES_Avg_Steps : 21.6
CS_Avg_Steps : 34.266666666666666
ASC_Avg_Steps : 18.066666666666666
ASC_ACC : 0.2


# How to Run the Code the get CoTs?